In [4]:
#!gsutil cp -r gs://bucket-for-kpf-project/UCI_Credit_Card.csv ./

In [1]:
!pip install --quiet --upgrade "kfp>=2,<3"
!pip install --quiet google.cloud
!pip install --quiet google-cloud-aiplatform
!pip install -U --quiet google-cloud-pipeline-components

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.5.3 which is incompatible.


In [1]:
import kfp
from google.cloud import aiplatform
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.custom_job import \
    CustomTrainingJobOp
from google_cloud_pipeline_components.v1.endpoint.create_endpoint.component import \
    endpoint_create as EndpointCreateOp
from google_cloud_pipeline_components.v1.endpoint.deploy_model.component import \
    model_deploy as ModelDeployOp

In [5]:
PROJECT_ID = "project-c8d59959-80e8-496f-96a"
BUCKET_URI = "gs://custom-trin-bucket"
DATASET_URI = "gs://bucket-for-kpf-project/UCI_Credit_Card.csv"
GCP_REGION = "us-central1"
DISPLAY_NAME = "credit-card-training-CUSTOM-09Jul26-1"
MACHINE_TYPE = "n1-standard-4"
TRAIN_URI = "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"
SERVING_URI = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"

In [22]:
def create_training_pipeline_custom_job_sample(
    replica_count: int = 1,
    machine_type: str = "MACHINE_TYPE",
    accelerator_type: str = "ACCELERATOR_TYPE_UNSPECIFIED",
    accelerator_count: int = 0,
    training_fraction_split: float = 0.8,
    validation_fraction_split: float = 0.1,
    test_fraction_split: float = 0.1,
    sync: bool = True,
):
    aiplatform.init(project=PROJECT_ID, location=GCP_REGION, staging_bucket=BUCKET_URI)

    job = aiplatform.CustomTrainingJob(
        display_name=DISPLAY_NAME,
        script_path="task.py",
        container_uri=TRAIN_URI,
        model_serving_container_image_uri=SERVING_URI,
    )

    dataset = dataset = aiplatform.TabularDataset('projects/90514214177/locations/us-central1/datasets/7580367688300494848')

    model = job.run(
        dataset=dataset,
        model_display_name=DISPLAY_NAME,
        #args=args,
        replica_count=replica_count,
        machine_type=MACHINE_TYPE,
        training_fraction_split=training_fraction_split,
        validation_fraction_split=validation_fraction_split,
        test_fraction_split=test_fraction_split,
        sync=sync,
    )

    model.wait()

    print(model.display_name)
    print(model.resource_name)
    print(model.uri)
    return model



In [23]:
create_training_pipeline_custom_job_sample()

Training script copied to:
gs://custom-trin-bucket/aiplatform-2026-07-09-14:31:31.494-aiplatform_custom_trainer_script-0.1.tar.gz.
Training Output directory:
gs://custom-trin-bucket/aiplatform-custom-training-2026-07-09-14:31:31.797 
View Training:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/3754186254363656192?project=90514214177
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/3754186254363656192 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/3754186254363656192 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/3754186254363656192 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/3754186254363656192 current state:
PIPELINE_STATE_RUNNING
CustomTrainingJob projects/90514214177/locations/us-central1/trainingPipelines/37541

resource name: projects/90514214177/locations/us-central1/models/5429248230225346560

In [7]:
aiplatform.init(project=PROJECT_ID, location=GCP_REGION, staging_bucket=BUCKET_URI)

In [24]:
endpoint = aiplatform.Endpoint.create(display_name=DISPLAY_NAME)

Creating Endpoint
Create Endpoint backing LRO: projects/90514214177/locations/us-central1/endpoints/5944883999280201728/operations/3413548339373801472
Endpoint created. Resource name: projects/90514214177/locations/us-central1/endpoints/5944883999280201728
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/90514214177/locations/us-central1/endpoints/5944883999280201728')


In [25]:
model = aiplatform.Model('projects/90514214177/locations/us-central1/models/5429248230225346560')

In [ ]:
endpoint.deploy(model,
                min_replica_count=1,
                max_replica_count=1,
                machine_type='n1-standard-4',
                accelerator_type=None,
                accelerator_count=None)

Deploying Model projects/90514214177/locations/us-central1/models/5429248230225346560 to Endpoint : projects/90514214177/locations/us-central1/endpoints/5944883999280201728
Deploy Endpoint model backing LRO: projects/90514214177/locations/us-central1/endpoints/5944883999280201728/operations/4387451758792671232


In [36]:
endpoint.predict(instances=[[6, 50000, 1, 1, 1, 57, -1, 1, -1, 0, 0, 0,
                            8617, 5670, 35835, 20940, 19146, 19131,
                            2000, 36681, 10000, 9000, 689, 679],
                            [6, 50000, 1, 1, 1, 23, -1, 1, -1, 0, 0, 0,
                            8617, 5670, 35835, 20940, 19146, 19131,
                            2000, 400, 2400, 200, 0, 0]])

Prediction(predictions=[[0.0], [1.0]], deployed_model_id='6921562586047578112', metadata=None, model_version_id='1', model_resource_name='projects/90514214177/locations/us-central1/models/5429248230225346560', explanations=None)